In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf
from scipy import stats
import warnings
import os

In [ ]:
def visualize_product(df_product, product_name):
    fig, axes = plt.subplots(3, 2, figsize=(14, 12))
    fig.suptitle(f"Exploratory Visualization - {product_name}", fontsize=14, fontweight="bold")

    # 1️⃣ Boxplot (All numeric columns)
    num_cols = df_product.select_dtypes(include=[np.number]).columns.tolist()
    sns.boxplot(data=df_product[num_cols], ax=axes[0, 0])
    axes[0, 0].set_title("Boxplot (All Numeric Columns)")
    axes[0, 0].set_xticklabels(num_cols, rotation=45, ha="right")

    # 2️⃣ Weekly Sales Trend
    axes[0, 1].plot(df_product.index, df_product["Sales"], color="orange", marker="o", linewidth=1)
    axes[0, 1].set_title("Weekly Sales Trend")
    axes[0, 1].set_xlabel("Date")
    axes[0, 1].set_ylabel("Sales")

    # 3️⃣ Sales Distribution
    sns.histplot(df_product["Sales"], bins=20, kde=True, ax=axes[1, 0], color="lightcoral")
    axes[1, 0].set_title("Sales Distribution (Histogram)")

    # 4️⃣ Rolling Mean & Std
    df_product["MA_4"] = df_product["Sales"].rolling(window=4, min_periods=1).mean()
    df_product["STD_4"] = df_product["Sales"].rolling(window=4, min_periods=1).std()
    axes[1, 1].plot(df_product.index, df_product["Sales"], label="Original", alpha=0.6)
    axes[1, 1].plot(df_product.index, df_product["MA_4"], label="4-Week MA", color="green", linewidth=2)
    axes[1, 1].fill_between(
        df_product.index,
        df_product["MA_4"] - df_product["STD_4"],
        df_product["MA_4"] + df_product["STD_4"],
        color="green", alpha=0.1
    )
    axes[1, 1].legend()
    axes[1, 1].set_title("Rolling Mean & Std (4 weeks)")

    # 5️⃣ ACF Plot
    n_obs = len(df_product["Sales"].dropna())
    max_lag = min(20, n_obs // 2)
    if max_lag > 1:
        plot_acf(df_product["Sales"].dropna(), ax=axes[2, 0], lags=max_lag, color="blue")
        axes[2, 0].set_title(f"Autocorrelation (ACF) - Lags={max_lag}")
    else:
        axes[2, 0].text(0.3, 0.5, "Data terlalu pendek untuk ACF", fontsize=10)
        axes[2, 0].set_axis_off()

    # 6️⃣ Q-Q Plot
    stats.probplot(df_product["Sales"].dropna(), dist="norm", plot=axes[2, 1])
    axes[2, 1].set_title("Q-Q Plot (Normality)")

    plt.tight_layout()
    plt.subplots_adjust(top=0.93)
    plt.show()

In [ ]:
def check_stationarity(series):
    series = series.dropna()
    # Cek apakah data konstan
    if series.nunique() == 1:
        return True, 0.0
    try:
        result = adfuller(series, autolag="AIC")
        return result[1] <= 0.05, result[1]
    except ValueError:
        return False, 0.0
    except Exception:
        return False, 0.0

def make_stationary(df_product):
    series = df_product["Sales"]
    is_stat, pval = check_stationarity(series)

    if not is_stat:
        df_product["Sales_Diff"] = series.diff()
        is_stat_diff, _ = check_stationarity(df_product["Sales_Diff"].dropna())
        if is_stat_diff:
            return df_product, "Sales_Diff", 1

        df_product["Sales_Diff2"] = df_product["Sales_Diff"].diff()
        return df_product, "Sales_Diff2", 2

    # Jika data konstan atau sudah stasioner
    return df_product, "Sales", 0

def clean_all_numeric(df_product):
    num_cols = df_product.select_dtypes(include=[np.number]).columns
    for col in num_cols:
        # Interpolasi missing values
        df_product[col] = df_product[col].interpolate(method="linear")
        # Deteksi & potong outlier
        Q1, Q3 = df_product[col].quantile([0.25, 0.75])
        IQR = Q3 - Q1
        lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
        df_product[col] = df_product[col].clip(lower, upper)
    return df_product

In [ ]:
def evaluate_product(df_product, original_series):
    mean_val = df_product["Sales"].mean()
    std_val = df_product["Sales"].std()

    # Outlier sebelum & sesudah
    Q1b, Q3b = original_series.quantile([0.25, 0.75])
    IQRb = Q3b - Q1b
    lowerb, upperb = Q1b - 1.5 * IQRb, Q3b + 1.5 * IQRb
    out_before = ((original_series < lowerb) | (original_series > upperb)).sum()

    Q1a, Q3a = df_product["Sales"].quantile([0.25, 0.75])
    IQRa = Q3a - Q1a
    lowera, uppera = Q1a - 1.5 * IQRa, Q3a + 1.5 * IQRa
    out_after = ((df_product["Sales"] < lowera) | (df_product["Sales"] > uppera)).sum()

    # ADF test (uji stasioneritas)
    try:
        adf_result = adfuller(df_product["Sales"].dropna(), autolag="AIC")
        p_value = round(adf_result[1], 6)
        if p_value <= 0.05:
            stationary = "Yes"
            d = 0
        else:
            stationary = "No"
            d = 1
    except Exception:
        stationary = "Error"
        p_value = None
        d = None

    return {
        "product": df_product["Produk"].iloc[0],
        "mean": mean_val,
        "std": std_val,
        "out_before": out_before,
        "out_after": out_after,
        "stationary": stationary,
        "d": d
    }

def summarize_preprocessing(results):
    total = len(results)
    stationary_count = sum(1 for r in results if r["stationary"] == "Yes")
    differencing_count = sum(1 for r in results if r["stationary"] == "No")
    constant_count = sum(1 for r in results if np.isclose(r["std"], 0))

    print("\n" + "=" * 52)
    print("PREPROCESSING SUMMARY:")
    print(f"• Total Produk: {total}")
    print(f"• Sudah Stasioner: {stationary_count}")
    print(f"• Perlu Differencing: {differencing_count}")
    print(f"• Data Konstan: {constant_count}")
    print("=" * 52)

In [ ]:
def main():
    print("="*80)
    print("AUTOMATIC PREPROCESSING + VISUALIZATION - MULTI PRODUCT")
    print("="*80)

    # Load dataset
    df = pd.read_excel("AVG 12W & 5W (W1-W40).xlsx", header=None)
    header_row = df[df.astype(str).apply(lambda x: x.str.contains("Kode Produk", case=False)).any(axis=1)].index[0]
    df = pd.read_excel("AVG 12W & 5W (W1-W40).xlsx", header=header_row)
    df = df.loc[:, ~df.columns.duplicated()]

    print("Kolom terbaca:", df.columns.tolist())

    # Ambil hanya kolom angka (minggu)
    week_cols = [c for c in df.columns if isinstance(c, (int, float))]
    print("Kolom minggu terdeteksi:", week_cols)

    # Transformasi ke format long
    df_long = df.melt(
        id_vars=["PRINC 1", "Kode Produk", "Produk"],
        value_vars=week_cols,
        var_name="Week",
        value_name="Sales"
    )

    # Tambahkan tanggal
    start_date = pd.Timestamp("2024-01-01")
    df_long["Date"] = df_long["Week"].apply(lambda x: start_date + pd.to_timedelta((int(x)-1)*7, unit="D"))
    df_long = df_long.sort_values(["Produk", "Date"])

    # Daftar produk
    products = df_long["Produk"].dropna().unique()
    total_products = len(products)
    print(f"Total Produk: {total_products}")

    results = []

    for i, prod in enumerate(products, 1):
        print(f"\n[{i}/{len(products)}] 🔍 Memproses produk: {prod}")
        df_product = df_long[df_long["Produk"] == prod].copy()
        df_product = df_product.set_index("Date").sort_index()
        original_series = df_product["Sales"].copy()

        # Cleaning & preprocessing otomatis
        df_product = clean_all_numeric(df_product)
        df_product, col, d = make_stationary(df_product)

        # Evaluasi & visualisasi otomatis
        evaluate_product(df_product, original_series)
        visualize_product(df_product, prod)
        status = summarize_arima_readiness(df_product, col, d)

        # Simpan hasil
        results.append({"Produk": prod, "Status": status})

    summarize_preprocessing(results)
    print("\n✅ Semua produk selesai diproses!")